# Linear-probe evaluation across datasets

Train a linear probe using a stratified train/test split from one dataset, then evaluate it on a second dataset held out in its entirety. The held-out dataset is never used for fitting or split selection. Use the first dataset's test metrics for within-dataset evaluation and the second dataset's validation metrics to assess transfer to a separate data source.

This notebook uses bacpipe's generated embeddings and ground-truth tables. It assumes each embedding row corresponds to the ground-truth row at the same position; the checks below fail if their lengths differ. Set the paths and parameters in the next cell before running.

In [ ]:
import json
import random
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from sklearn.metrics import ConfusionMatrixDisplay, accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split

import bacpipe
from classifier.train_probe import train_probe

MODEL_NAME = "aves_especies"
TRAIN_TEST_AUDIO_DIR = Path(r"C:\path\to\train_test_dataset")
VALIDATION_AUDIO_DIR = Path(r"C:\path\to\held_out_validation_dataset")
ANNOTATIONS_FILENAME = "annotations.csv"
RESULTS_DIR = Path("bacpipe_results") / "evalution_pipeline"

TEST_SIZE = 0.2
SEED = 42
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
BATCH_SIZE = 64
LEARNING_RATE = 0.001
NUM_EPOCHS = 20

if not 0 < TEST_SIZE < 1:
    raise ValueError("TEST_SIZE must be between 0 and 1.")

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print(f"Device: {DEVICE}")
print(f"Train/test dataset: {TRAIN_TEST_AUDIO_DIR}")
print(f"Held-out validation dataset: {VALIDATION_AUDIO_DIR}")

## Load both datasets

The first dataset supplies the training and test examples. The second dataset is loaded independently and remains untouched until validation. Both must use the same label scheme and the same embedding model.

In [ ]:
def load_dataset(audio_dir):
    audio_dir = Path(audio_dir)
    embeddings = bacpipe.run_pipeline_for_single_model(
        model_name=MODEL_NAME,
        audio_dir=str(audio_dir),
        check_if_already_processed=True,
        only_embed_annotations=False,
    ).embeddings(return_type="array")
    ground_truth = bacpipe.ground_truth_by_model(
        model=MODEL_NAME,
        audio_dir=str(audio_dir),
        annotations_filename=ANNOTATIONS_FILENAME,
        main_results_dir=str(RESULTS_DIR),
        overwrite=False,
        only_embed_annotations=False,
    )

    embeddings = np.asarray(embeddings)
    if embeddings.ndim != 2:
        raise ValueError(f"Expected a 2D embedding array, got {embeddings.shape}.")
    if len(embeddings) != len(ground_truth):
        raise ValueError(
            f"Embedding/ground-truth row mismatch for {audio_dir}: "
            f"{len(embeddings)} embeddings vs {len(ground_truth)} labels."
        )
    return embeddings, ground_truth.reset_index(drop=True)


train_test_embeddings, train_test_ground_truth = load_dataset(TRAIN_TEST_AUDIO_DIR)
validation_embeddings, validation_ground_truth = load_dataset(VALIDATION_AUDIO_DIR)

print("Train/test dataset:", train_test_embeddings.shape, train_test_ground_truth.shape)
print("Held-out dataset:", validation_embeddings.shape, validation_ground_truth.shape)

## Prepare single-label examples

Only rows with exactly one active label (`simultaneous_labels == 1`) are used for this multiclass probe. The generated embedding rows are filtered with the same mask as their ground-truth table.

In [ ]:
NON_LABEL_COLUMNS = {"start", "end", "audiofilename", "simultaneous_labels"}


def single_label_examples(embeddings, ground_truth):
    if "simultaneous_labels" not in ground_truth.columns:
        raise KeyError("Ground truth must contain a 'simultaneous_labels' column.")
    label_columns = [column for column in ground_truth.columns if column not in NON_LABEL_COLUMNS]
    if not label_columns:
        raise ValueError("No label columns were found in the ground-truth table.")

    mask = ground_truth["simultaneous_labels"].eq(1).to_numpy()
    selected_labels = ground_truth.loc[mask, label_columns]
    labels_per_row = selected_labels.eq(1).sum(axis=1)
    if not labels_per_row.eq(1).all():
        raise ValueError("Rows marked as single-label must have exactly one active label.")
    labels = selected_labels.idxmax(axis=1).to_numpy()
    selected_embeddings = np.asarray(embeddings)[mask]
    if len(selected_embeddings) != len(labels):
        raise ValueError("Filtered embedding and label counts do not match.")
    if not len(labels):
        raise ValueError("No single-label examples were found.")
    return selected_embeddings, labels


X_train_test, y_train_test = single_label_examples(
    train_test_embeddings, train_test_ground_truth
)
X_validation, y_validation = single_label_examples(
    validation_embeddings, validation_ground_truth
)

print(f"Train/test source: {len(y_train_test)} single-label examples")
print(f"Held-out validation source: {len(y_validation)} single-label examples")

## Split and train

The train/test split is stratified by class and reproducible with `SEED`. The second dataset is not passed to the split or training functions.

In [ ]:
label_names = sorted(np.unique(y_train_test).tolist())
label2index = {label: index for index, label in enumerate(label_names)}
y_train_test_index = np.asarray([label2index[label] for label in y_train_test])

class_counts = pd.Series(y_train_test).value_counts()
if (class_counts < 2).any():
    raise ValueError(
        "Stratified train/test splitting requires at least two examples per class. "
        f"Insufficient classes: {class_counts[class_counts < 2].to_dict()}"
    )

all_indices = np.arange(len(y_train_test_index))
train_indices, test_indices = train_test_split(
    all_indices,
    test_size=TEST_SIZE,
    random_state=SEED,
    stratify=y_train_test_index,
)

split_table = pd.DataFrame({"label": y_train_test})
split_table["predefined_set"] = "unused"
split_table.loc[train_indices, "predefined_set"] = "train"
split_table.loc[test_indices, "predefined_set"] = "test"
display(pd.crosstab(split_table["label"], split_table["predefined_set"]))

probe = train_probe(
    embeds=X_train_test,
    df=split_table,
    label2index=label2index,
    config="linear",
    learning_rate=LEARNING_RATE,
    num_epochs=NUM_EPOCHS,
    device=DEVICE,
    batch_size=BATCH_SIZE,
)
print("Linear probe training complete.")

## Evaluate on test and held-out validation

Report the train/test dataset's test split separately from the external validation dataset. Validation labels must be represented in the training label mapping; otherwise the notebook stops rather than silently dropping classes.

In [ ]:
unknown_validation_labels = sorted(set(y_validation) - set(label2index))
if unknown_validation_labels:
    raise ValueError(
        "Held-out validation contains labels absent from the training dataset: "
        f"{unknown_validation_labels}"
    )

y_test = y_train_test_index[test_indices]
probe = probe.to(DEVICE)
probe.eval()

def predict_labels(embeddings):
    inputs = torch.as_tensor(embeddings, dtype=torch.float32, device=DEVICE)
    with torch.inference_mode():
        return probe(inputs).argmax(dim=1).cpu().numpy()


y_test_pred = predict_labels(X_train_test[test_indices])
y_validation_index = np.asarray([label2index[label] for label in y_validation])
y_validation_pred = predict_labels(X_validation)
class_ids = list(range(len(label_names)))

test_report = classification_report(
    y_test,
    y_test_pred,
    labels=class_ids,
    target_names=label_names,
    output_dict=True,
    zero_division=0,
)
validation_report = classification_report(
    y_validation_index,
    y_validation_pred,
    labels=class_ids,
    target_names=label_names,
    output_dict=True,
    zero_division=0,
)

print(f"Train/test source — test accuracy: {accuracy_score(y_test, y_test_pred):.4f}")
display(pd.DataFrame(test_report).T)
print(f"Held-out source — validation accuracy: {accuracy_score(y_validation_index, y_validation_pred):.4f}")
display(pd.DataFrame(validation_report).T)

fig, axes = plt.subplots(1, 2, figsize=(max(12, len(label_names)), 5))
ConfusionMatrixDisplay(
    confusion_matrix(y_test, y_test_pred, labels=class_ids),
    display_labels=label_names,
).plot(ax=axes[0], xticks_rotation=45, colorbar=False)
axes[0].set_title("Test split")
ConfusionMatrixDisplay(
    confusion_matrix(y_validation_index, y_validation_pred, labels=class_ids),
    display_labels=label_names,
).plot(ax=axes[1], xticks_rotation=45, colorbar=False)
axes[1].set_title("Held-out validation dataset")
fig.tight_layout()
plt.show()

## Save probe and run metadata

Outputs are written without deleting existing results. The metadata records both dataset paths and the train/test settings so the evaluation can be reproduced.

In [ ]:
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
torch.save(probe.state_dict(), RESULTS_DIR / "linear_probe.pt")
(RESULTS_DIR / "label2index.json").write_text(
    json.dumps(label2index, indent=2), encoding="utf-8"
)

run_metadata = {
    "model_name": MODEL_NAME,
    "train_test_audio_dir": str(TRAIN_TEST_AUDIO_DIR),
    "held_out_validation_audio_dir": str(VALIDATION_AUDIO_DIR),
    "annotations_filename": ANNOTATIONS_FILENAME,
    "test_size": TEST_SIZE,
    "seed": SEED,
    "device": DEVICE,
    "batch_size": BATCH_SIZE,
    "learning_rate": LEARNING_RATE,
    "num_epochs": NUM_EPOCHS,
    "label2index": label2index,
    "test_report": test_report,
    "held_out_validation_report": validation_report,
}
(RESULTS_DIR / "evaluation_metrics.json").write_text(
    json.dumps(run_metadata, indent=2), encoding="utf-8"
)
print(f"Saved probe and metrics to {RESULTS_DIR.resolve()}")